In [ ]:
# ==========================================
#  DAILY STOCK RANGE SCANNER (Fixed Version)
#  Yahoo Primary → Stooq Fast Free Fallback
# ==========================================

!pip install yfinance pandas_datareader -q

import yfinance as yf
import pandas as pd
import warnings
from datetime import datetime, timedelta
import time
import pandas_datareader.data as web

warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# SETTINGS
# ==========================================
LOOKBACK_DAYS = 80
TOP_PCT       = 25
BTM_PCT       = 25
MAX_RESULTS   = 30

MIN_PRICE     = 25
MAX_PRICE     = 500

# ==========================================
# TICKER LIST - ONE PER LINE (Important!)
# ==========================================
RAW_LIST = """
A
AA
AAL
AAP
AAPL
ABBV
ABNB
ABT
ACN
ADBE
ADI
ADM
ADP
ADSK
AEP
AES
AFL
AFRM
AGNC
AI
AIG
AKAM
ALB
ALK
ALL
AMAT
AMD
AMGN
AMRN
AMZN
APA
APH
APTV
AVGO
AXP
BA
BABA
BAC
BAX
BBY
BEN
BIDU
BIIB
BITO
BK
BKR
BMY
BP
BSX
BX
BYND
C
CAH
CAT
CB
CCI
CCJ
CCL
CF
CFG
CHWY
CI
CL
CLF
CLX
CMCSA
CME
CNC
CNP
COF
COIN
COP
COST
CPB
CPRI
CRM
CRON
CRWD
CSCO
CSX
CTVA
CVNA
CVS
CVX
CZR
D
DAL
DD
DE
DELL
DHI
DHR
DIA
DIS
DKNG
DLR
DOCU
DOW
DRI
DVN
DXC
EA
EBAY
ED
EEM
EFA
EIX
EL
EMR
ENPH
EOG
EQR
EQT
ETSY
EVRG
EW
EWJ
EWW
EWY
EWZ
EXC
EXPE
F
FANG
FAST
FCX
FDX
FE
FI
FITB
FOXA
FSLR
FTI
FTV
FXE
FXI
GD
GDX
GE
GILD
GLD
GLW
GM
GME
GOOG
GOOGL
GPRO
GPS
GS
HAL
HBAN
HBI
HCA
HD
HIG
HLT
HOG
HOLX
HON
HPE
HPQ
HYG
IBB
IBIT
IBM
ICE
IEF
INTC
IP
IPG
IRM
IVZ
IWM
IYR
JCI
JD
JETS
JNJ
JNPR
JPM
K
KHC
KMI
KO
KR
KRE
KSS
KWEB
LEN
LI
LKQ
LLY
LNC
LOW
LQD
LUMN
LUV
LVS
LYB
LYFT
MA
MAR
MARA
MAS
MCD
MCHP
MDLZ
MDT
MET
META
MGM
MMM
MO
MOS
MPC
MRK
MRNA
MRVL
MS
MSFT
MSTR
MTB
MTCH
MU
NCLH
NEE
NEM
NET
NFLX
NKE
NOW
NRG
NTAP
NTES
NVAX
NVDA
NWL
NWSA
O
OIH
OKE
OMC
ON
ORCL
OXY
PARA
PBR
PDD
PENN
PEP
PFE
PFG
PG
PGR
PINS
PLTR
PNC
PPL
PRU
PSX
PTON
PYPL
QCOM
QQQ
RBLX
RCL
RF
RIG
RIOT
RIVN
ROKU
ROST
RTX
RUN
SBUX
SCHD
SCHW
SEDG
SHOP
SIRI
SLB
SLV
SMCI
SMH
SNAP
SNOW
SO
SOFI
SOXL
SOXS
SPG
SPX
SPXL
SPXS
SPY
SQQQ
STX
SWKS
SYF
SYY
T
TAP
TBT
TCOM
TDOC
TFC
TGT
TJX
TLT
TMO
TMUS
TPR
TQQQ
TRIP
TRV
TSLA
TSM
TSN
TT
TTD
TTWO
TXN
U
UA
UAA
UAL
UBER
ULTA
UNG
UNH
UNP
UPS
UPST
URBN
USB
USO
UVXY
V
VALE
VFC
VGK
VTR
VXX
VZ
WAB
WBA
WBD
WDC
WFC
WM
WMB
WMT
WU
WY
WYNN
X
XBI
XEL
XHB
XLB
XLC
XLE
XLF
XLI
XLK
XLP
XLRE
XLU
XLV
XLY
XOM
XOP
XRT
XRX
XSP
YELP
YINN
ZION
ZM
"""

# ==========================================
# FUNCTIONS
# ==========================================
def get_clean_tickers():
    tickers = [t.strip() for t in RAW_LIST.split('\n') if t.strip()]
    return list(set(tickers))  # Remove duplicates

def get_yahoo_data(tickers, days=150):
    print(f"   [YAHOO] Downloading {len(tickers)} symbols...")
    start_date = datetime.now() - timedelta(days=days)
    # Map tickers like BRK.B → BRK-B
    mapping = {t.replace(".", "-"): t for t in tickers}
    yahoo_tickers = list(mapping.keys())
    
    try:
        df = yf.download(yahoo_tickers, start=start_date, progress=False, timeout=30, threads=True)
        closes = df['Close'] if isinstance(df, pd.DataFrame) and 'Close' in df.columns else df
        closes = closes.rename(columns=mapping)
        return closes.sort_index()
    except Exception as e:
        print(f"   [YAHOO] Failed: {e}")
        return pd.DataFrame()

def get_stooq_data(tickers, days=150):
    print(f"   [STOOQ] Downloading {len(tickers)} symbols...")
    try:
        start = datetime.now() - timedelta(days=days)
        batch_size = 60
        all_data = {}
        
        for i in range(0, len(tickers), batch_size):
            batch = tickers[i:i+batch_size]
            try:
                df = web.DataReader(batch, 'stooq', start=start)
                if not df.empty:
                    close_df = df['Close'] if 'Close' in df.columns else df
                    if isinstance(close_df, pd.DataFrame):
                        for col in close_df.columns:
                            all_data[col] = close_df[col]
            except:
                pass
            time.sleep(0.5)
        
        if all_data:
            closes = pd.DataFrame(all_data)
            print(f"   [STOOQ] Success - Got {len(closes.columns)} tickers")
            return closes.sort_index()
    except Exception as e:
        print(f"   [STOOQ] Failed: {e}")
    return pd.DataFrame()

# ==========================================
# MAIN
# ==========================================
def run_scanner():
    print("🚀 Starting Daily Stock Range Scanner...")
    print("="*70)
    
    tickers = get_clean_tickers()
    print(f"Loaded {len(tickers)} unique symbols.\n")
    
    # Try Yahoo first
    data = get_yahoo_data(tickers)
    
    # Fallback to Stooq
    if data.empty or len(data) < 50:
        print("\n⚠️ Yahoo failed → Trying Stooq...")
        data = get_stooq_data(tickers)
    
    if data.empty or len(data) < 30:
        print("\n❌ Both Yahoo and Stooq failed. Please try again later.")
        return
    
    print(f"\n✅ Successfully loaded data for {len(data.columns)} tickers.\n")
    
    # Analysis
    recent_data = data.tail(LOOKBACK_DAYS)
    premium_list = []
    discount_list = []
    filtered_out = 0
    
    for col in recent_data.columns:
        try:
            series = recent_data[col].dropna()
            if len(series) < 25:
                continue
            curr = float(series.iloc[-1])
            if curr < MIN_PRICE or curr > MAX_PRICE:
                filtered_out += 1
                continue
            hi = float(series.max())
            lo = float(series.min())
            rng = hi - lo
            if rng == 0: 
                continue
            loc = (curr - lo) / rng * 100
            
            item = {'ticker': col, 'loc': round(loc, 1), 'price': round(curr, 2)}
            
            if loc >= (100 - TOP_PCT):
                premium_list.append(item)
            elif loc <= BTM_PCT:
                discount_list.append(item)
        except:
            continue

    print(f"Skipped {filtered_out} tickers outside price range.\n")

    discount_list.sort(key=lambda x: x['loc'])
    premium_list.sort(key=lambda x: x['loc'], reverse=True)
    
    final_discount = discount_list[:MAX_RESULTS]
    final_premium = premium_list[:MAX_RESULTS]

    # Output
    print("="*70)
    print(f"MOST DISCOUNTED (Bottom {BTM_PCT}% of {LOOKBACK_DAYS}-day range)")
    print("="*70)
    for s in final_discount:
        print(f"{s['ticker']:<8} {s['loc']:>6.1f}%   ${s['price']:>8.2f}")

    print("\n" + "="*70)
    print(f"MOST PREMIUM (Top {TOP_PCT}% of {LOOKBACK_DAYS}-day range)")
    print("="*70)
    for s in final_premium:
        print(f"{s['ticker']:<8} {s['loc']:>6.1f}%   ${s['price']:>8.2f}")

    print("\n" + "#"*80)
    print("TRADINGVIEW IMPORT STRINGS")
    print("#"*80)
    print(f"\nDISCOUNT ({len(final_discount)}):")
    print(", ".join(x['ticker'] for x in final_discount))
    print(f"\nPREMIUM ({len(final_premium)}):")
    print(", ".join(x['ticker'] for x in final_premium))

    print("\n✅ Done! Run this script every market day.")

# Run it
run_scanner()